In [1]:
import numpy as np
import pandas as pd

In [5]:
# read the 2 hour auctions-impression-click data
data = pd.read_csv(r"C:\Users\Chand\Downloads\pranjal_data\Mingao Analysis\Topsort-Online-Ads\2-hour auctions.csv")

In [11]:
import pandas as pd

# 1. Inspect data types
print("=== Column Data Types ===")
print(data.dtypes)
print()

# 2. Check a few unique values for non-numeric columns
print("=== Sample Unique Values for Object Columns ===")
for col in data.select_dtypes(include='object').columns:
    print(f"\n{col}:")
    print(data[col].dropna().unique()[:5])  # show first 5 unique non-null values
print()

# 3. Check missing value summary
print("=== Missing Values Summary ===")
print(data.isna().mean().sort_values(ascending=False))

# 4. Optional cleaning for time columns
# Convert to datetime if not already
for tcol in ['IMPRESSION_TIME', 'CLICK_TIME', 'AUCTION_TIME']:
    if tcol in data.columns:
        data[tcol] = pd.to_datetime(data[tcol], errors='coerce')

# 5. Re-check the data types after cleaning
print("\n=== Data Types After Cleaning ===")
print(data.dtypes)


=== Column Data Types ===
AUCTION_ID_N        object
CAMPAIGN_ID_N       object
PRODUCT_ID_N        object
VENDOR_ID_N         object
AUCTION_TIME        object
RANKING              int64
IS_WINNER             bool
QUALITY            float64
FINAL_BID            int64
AUCTION_PRICE      float64
CONVERSION_RATE    float64
PACING             float64
OPAQUE_USER_ID      object
PLACEMENT            int64
IMPRESSION_TIME     object
CLICK_TIME          object
CLICK_ID            object
USER_ID             object
dtype: object

=== Sample Unique Values for Object Columns ===

AUCTION_ID_N:
['068433b09e587397b204f2a9c698eb52' '068433bab22471318204040385b31dd6'
 '068433b9405f7e5e8c04d060e05476a7' '068433c064a17de5a904066571df3ba6'
 '068433c2f9677c7284044af84c4b1c56']

CAMPAIGN_ID_N:
['0197376ba2007373bf77b22c32958d78' '019736bbda8770b3b535275699cfb002'
 '019733b2918078729736df577ee2b8f5' '01972c5be80470b2bf81ec0230f79685'
 '01973ca363aa7c70bf0c141de74c4563']

PRODUCT_ID_N:
['66bba1874fab7bd0bbe

In [16]:
# --- 1. Filter only rows where ad was shown (winner) and has an impression ---
df_ctr = data[(data['IS_WINNER'] == True) & (data['IMPRESSION_TIME'].notna())].copy()

# --- 2. Create an indicator for whether a click occurred ---
df_ctr['HAS_CLICK'] = df_ctr['CLICK_TIME'].notna().astype(int)

# --- 3. Group by ranking and compute CTR (mean of HAS_CLICK) ---
ctr_by_rank = (
    df_ctr.groupby('RANKING')['HAS_CLICK']
    .mean()
    .reset_index(name='CTR')
    .sort_values('RANKING')
)

# --- 4. Display ---
print(ctr_by_rank)

    RANKING       CTR
0         1  0.028598
1         2  0.022412
2         3  0.025241
3         4  0.025018
4         5  0.024821
5         6  0.025161
6         7  0.024736
7         8  0.024286
8         9  0.023325
9        10  0.023746
10       11  0.022727
11       12  0.023937
12       13  0.025506
13       14  0.021939
14       15  0.021369
15       16  0.022445
16       17  0.021532
17       18  0.021338
18       19  0.022154
19       20  0.023010
20       21  0.022853
21       22  0.020533
22       23  0.021267
23       24  0.020992
24       25  0.022088
25       26  0.021086
26       27  0.022853
27       28  0.019288
28       29  0.022772
29       30  0.020575
30       31  0.020856
31       32  0.020603
32       33  0.019118
33       34  0.020513
34       35  0.020527
35       36  0.021952
36       37  0.022504
37       38  0.018862
38       39  0.020503
39       40  0.020741
40       41  0.005096
41       42  0.004946
42       43  0.006530
43       44  0.003750
44       4

In [10]:
# Assuming `data` is your DataFrame
prop_impression = data['IMPRESSION_TIME'].notna().mean()
prop_click = data['CLICK_TIME'].notna().mean()

print(f"Proportion with IMPRESSION_TIME not null: {prop_impression:.4f}")
print(f"Proportion with CLICK_TIME not null: {prop_click:.4f}")

Proportion with IMPRESSION_TIME not null: 0.1515
Proportion with CLICK_TIME not null: 0.0036


In [24]:
from matplotlib import pyplot as plt

plt.plot(ctr_by_rank['RANKING'], ctr_by_rank['CTR'], marker='o')
plt.xlabel("Ranking (slot position)")
plt.ylabel("Click-Through Rate (CTR)")
plt.title("CTR by Auction Ranking (Conditioned on Impression)")
plt.show()


ModuleNotFoundError: No module named 'matplotlib.backends.registry'